# Lab opt_01 — Optimizers from scratch: GD, momentum, Adam, Newton

**Learning goals**
1. Implement gradient descent, heavy-ball momentum, Adam and Newton's method in numpy.
2. See the stability limit $\eta < 2/\lambda_{\max}$ and the role of the condition number $\kappa$ on quadratics.
3. Compare iterations-to-converge on the ill-conditioned **Rosenbrock** valley.

Do the paper homework for *Gradient descent* first.

**Reading:** [Boyd & Vandenberghe, *Convex Optimization* (free)](https://web.stanford.edu/~boyd/cvxbook/) §9.2–9.5 ·
[Kingma & Ba, *Adam* (arXiv:1412.6980)](https://arxiv.org/abs/1412.6980) Algorithm 1 ·
[MIT 18.065](https://ocw.mit.edu/courses/18-065-matrix-methods-in-data-analysis-signal-processing-and-machine-learning-spring-2018/) lectures on gradient descent & momentum.

Every optimizer returns the **whole path**: a numpy array of shape `(n_steps + 1, d)` whose row 0 is `x0`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1. Test problems

**Quadratic** $f(x) = \tfrac12 x^\top A x - b^\top x$, $\nabla f = Ax - b$, $\nabla^2 f = A$, minimiser $x^\star = A^{-1}b$.

**Rosenbrock** $f(x,y) = (1-x)^2 + 100(y - x^2)^2$, minimum $f(1,1) = 0$ at the bottom of a long curved valley.

Implement `quadratic(A, b)` returning the three functions `(f, grad, hess)`, and `rosenbrock`, `rosenbrock_grad`, `rosenbrock_hess`
(derive the derivatives on paper first).

In [ ]:
def quadratic(A, b):
    """Return (f, grad, hess) for f(x) = 1/2 x^T A x - b^T x."""
    # TODO
    raise NotImplementedError


def rosenbrock(x):
    # TODO
    raise NotImplementedError


def rosenbrock_grad(x):
    # TODO
    raise NotImplementedError


def rosenbrock_hess(x):
    # TODO
    raise NotImplementedError

## 2. Gradient descent

$x_{t+1} = x_t - \eta\nabla f(x_t)$. On a quadratic the error evolves as $e_{t+1} = (I - \eta A)e_t$, so each eigen-direction
is multiplied by $1 - \eta\lambda_i$: GD converges iff $0 < \eta < 2/\lambda_{\max}$; the best fixed step is
$\eta^\star = 2/(\lambda_{\min}+\lambda_{\max})$, giving contraction $\rho = \frac{\kappa-1}{\kappa+1}$.

Also implement `iters_to_converge(path, x_star, tol)`: the first index $k$ with $\|x_k - x^\star\| < \text{tol}$, or `-1` if never.

In [ ]:
def gd(grad, x0, lr, n_steps):
    # TODO
    raise NotImplementedError


def iters_to_converge(path, x_star, tol):
    # TODO
    raise NotImplementedError

In [ ]:
A = np.array([[10.0, 0.0], [0.0, 1.0]]); b = np.array([1.0, 1.0])
f, grad, hess = quadratic(A, b)
x_star = np.linalg.solve(A, b)
lmax = 10.0
for lr in [0.5 / lmax, 1.0 / lmax, 2 / 11, 1.9 / lmax, 2.05 / lmax]:
    p = gd(grad, [-1.0, 2.0], lr, 300)
    print(f"lr = {lr:.4f}  ({lr * lmax:.2f}/lambda_max): iters to 1e-6 = {iters_to_converge(p, x_star, 1e-6):>4}, "
          f"final error {np.linalg.norm(p[-1] - x_star):.2e}")

In [ ]:
# Contour plot of the paths
def contour(f, xlim, ylim, levels=30):
    X, Y = np.meshgrid(np.linspace(*xlim, 200), np.linspace(*ylim, 200))
    Z = np.array([[f(np.array([u, v])) for u, v in zip(r1, r2)] for r1, r2 in zip(X, Y)])
    plt.contour(X, Y, Z, levels=levels, cmap="Greys")

contour(f, (-1.2, 1.2), (-1, 2.5))
for lr in [0.05, 0.18, 0.199]:
    p = gd(grad, [-1.0, 2.0], lr, 40)
    plt.plot(p[:, 0], p[:, 1], ".-", label=f"lr={lr}")
plt.legend(); plt.title("GD zig-zags as lr approaches 2/lambda_max"); plt.show()

## 3. Momentum

Heavy ball (the convention used in the paper problems):
$$v_{t+1} = \beta v_t + \nabla f(x_t),\qquad x_{t+1} = x_t - \eta v_{t+1},\qquad v_0 = 0.$$
On a long slope the effective step becomes $\eta/(1-\beta)$; on ill-conditioned quadratics the best tuned momentum
improves the rate from $\frac{\kappa-1}{\kappa+1}$ to about $\frac{\sqrt\kappa-1}{\sqrt\kappa+1}$.

In [ ]:
def momentum(grad, x0, lr, beta, n_steps):
    # TODO
    raise NotImplementedError

## 4. Adam

With $g_t = \nabla f(x_{t-1})$:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2,\quad
\hat m_t = \frac{m_t}{1-\beta_1^t},\quad \hat v_t = \frac{v_t}{1-\beta_2^t},\quad x_t = x_{t-1} - \eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$
All operations are element-wise. Note: the first step moves each coordinate by (almost exactly) $\eta$.

In [ ]:
def adam(grad, x0, lr, n_steps, beta1=0.9, beta2=0.999, eps=1e-8):
    # TODO
    raise NotImplementedError

## 5. Newton's method

$x_{t+1} = x_t - [\nabla^2 f(x_t)]^{-1}\nabla f(x_t)$ — use `np.linalg.solve`, never form the inverse.
Exact in one step on a quadratic; quadratically convergent near a minimum with positive-definite Hessian.

In [ ]:
def newton(grad, hess, x0, n_steps):
    # TODO
    raise NotImplementedError

## 6. Race on Rosenbrock
Start at the classic point $(-1.2, 1)$. Tune the learning rates yourself — what is the largest stable GD step?

In [ ]:
x0 = [-1.2, 1.0]; star = np.array([1.0, 1.0])
runs = {
    "GD lr=1e-3": gd(rosenbrock_grad, x0, 1e-3, 20_000),
    "momentum lr=1e-4, beta=0.9": momentum(rosenbrock_grad, x0, 1e-4, 0.9, 20_000),
    "Adam lr=1e-2": adam(rosenbrock_grad, x0, 1e-2, 20_000),
    "Newton": newton(rosenbrock_grad, rosenbrock_hess, x0, 50),
}
for name, p in runs.items():
    print(f"{name:<28} iters to 1e-3: {iters_to_converge(p, star, 1e-3):>6}   final f = {rosenbrock(p[-1]):.2e}")

contour(rosenbrock, (-1.5, 1.5), (-0.5, 1.5), levels=np.logspace(-2, 3, 25))
for name, p in runs.items():
    plt.plot(p[:, 0], p[:, 1], ".-", ms=2, label=name)
plt.plot(1, 1, "k*", ms=12); plt.legend(fontsize=8); plt.show()

In [ ]:
# Your turn: iterations-to-converge vs condition number for GD (optimal step) and momentum
for kappa in [2, 10, 100]:
    Ak = np.diag([1.0, float(kappa)]); _, gk, _ = quadratic(Ak, np.zeros(2))
    p_gd = gd(gk, [1.0, 1.0], 2 / (1 + kappa), 5000)
    beta = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2
    p_hb = momentum(gk, [1.0, 1.0], (2 / (1 + np.sqrt(kappa))) ** 2, beta, 5000)
    print(f"kappa={kappa:>4}: GD {iters_to_converge(p_gd, np.zeros(2), 1e-8):>5}   momentum {iters_to_converge(p_hb, np.zeros(2), 1e-8):>5}")

## Grade
Runs the hidden checks (fresh inputs, fixed seeds) and records the result in the Learning Center.

In [ ]:
from lc_client import grade
grade("opt_01_optimizers", globals())